# Credit Scoring Model
### CodeAlpha Machine Learning Internship
**Name:** Sumit Kumar Mahto &nbsp;|&nbsp; **ID:** CA/DF1/47253

The idea here is pretty simple — given someone's financial history, can we predict whether they'll repay a loan or not? This is a classic binary classification problem and something banks do every day.

I'll try three different algorithms and see which one works best.


In [ ]:
# all the libraries i need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix)
import joblib

print("imports done!")

## Step 1 — Load the data

I'm using a synthetic dataset here to keep things self-contained. If you want to use real data, just uncomment the `pd.read_csv` line and point it to your file.

In [ ]:
# uncomment this when using real data
# df = pd.read_csv("credit_data.csv")

# synthetic data for now
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    "annual_income":    np.random.randint(100000, 2000000, n),
    "loan_amount":      np.random.randint(50000, 1000000, n),
    "loan_tenure":      np.random.choice([12, 24, 36, 48, 60], n),
    "num_credit_lines": np.random.randint(1, 10, n),
    "missed_payments":  np.random.randint(0, 6, n),
    "employment_years": np.random.randint(0, 20, n),
    "debt_to_income":   np.round(np.random.uniform(0.05, 0.90, n), 2),
    "credit_score":     np.random.randint(300, 900, n),
    "age":              np.random.randint(21, 65, n),
    # 1 means they'll repay, 0 means high risk
    "creditworthy":     np.random.choice([0, 1], n, p=[0.35, 0.65])
})

print(f"dataset shape: {df.shape}")
df.head()

In [ ]:
# quick check — any missing values? class balance?
print("missing values:")
print(df.isnull().sum())
print()
print("class split:")
print(df["creditworthy"].value_counts())
print(f"
positive rate: {df['creditworthy'].mean():.1%}")

In [ ]:
# visualise the distributions — always good to look at your data first
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Feature Distributions", fontsize=13)

cols = ["annual_income", "loan_amount", "credit_score", "debt_to_income",
        "missed_payments", "employment_years", "age", "loan_tenure"]

for ax, col in zip(axes.flatten(), cols):
    df[col].hist(ax=ax, bins=20, color="#4CAF50", edgecolor="white", alpha=0.8)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("count")

plt.tight_layout()
plt.show()

## Step 2 — Feature Engineering

Raw features are okay but I want to create a couple of derived features that might be more informative for the model.

In [ ]:
# loan to income ratio — a high value means someone is borrowing too much relative to earnings
df["loan_to_income"] = df["loan_amount"] / df["annual_income"]

# reliability score — rewards good credit history, penalises missed payments
df["reliability_score"] = df["credit_score"] / (df["missed_payments"] + 1)

print("new features added!")
df[["loan_to_income", "reliability_score"]].describe()

## Step 3 — Split & Preprocess

I'm using stratified split so the class ratio stays consistent in train and test sets. Then scaling the features since logistic regression is sensitive to scale.

In [ ]:
features = [
    "annual_income", "loan_amount", "loan_tenure", "num_credit_lines",
    "missed_payments", "employment_years", "debt_to_income", "credit_score",
    "age", "loan_to_income", "reliability_score"
]

X = df[features]
y = df["creditworthy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"training size : {X_train.shape[0]}")
print(f"testing size  : {X_test.shape[0]}")

# pipeline handles imputing + scaling together cleanly
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

## Step 4 — Train the Models

I'll try Logistic Regression, Decision Tree and Random Forest. Using 5-fold cross-validation to get a fair comparison before touching the test set.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("pre", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("pre", preprocessor),
        ("clf", DecisionTreeClassifier(max_depth=6, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("cross-validation ROC-AUC scores:")
print("-" * 40)
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{name:<25} {scores.mean():.4f} ± {scores.std():.4f}")

## Step 5 — Test Set Evaluation

Now let's see how each model does on unseen data.

In [ ]:
results = []
trained = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    trained[name] = pipeline

    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_proba), 4),
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df

## Step 6 — Plots

Three useful visualisations — ROC curves, confusion matrix and feature importance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Credit Scoring Model — Results", fontsize=13, fontweight="bold")

# roc curves for all 3 models
ax = axes[0]
colors = ["#2196F3", "#4CAF50", "#FF5722"]
for (name, pipeline), color in zip(trained.items(), colors):
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})", color=color, lw=2)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set_title("ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# confusion matrix for best model
ax = axes[1]
best_name = results_df["ROC-AUC"].idxmax()
cm = confusion_matrix(y_test, trained[best_name].predict(X_test))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["High Risk", "Creditworthy"],
            yticklabels=["High Risk", "Creditworthy"])
ax.set_title(f"Confusion Matrix
({best_name})")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")

# feature importances from random forest
ax = axes[2]
rf_imp = pd.Series(
    trained["Random Forest"].named_steps["clf"].feature_importances_,
    index=features
).sort_values()
rf_imp.tail(8).plot(kind="barh", ax=ax, color="#4CAF50", alpha=0.85)
ax.set_title("Feature Importances
(Random Forest)")
ax.set_xlabel("importance score")
ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("task1_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved as task1_results.png")

## Step 7 — Save the Best Model

In [ ]:
joblib.dump(trained[best_name], "credit_model.pkl")
print(f"saved: credit_model.pkl  (best model: {best_name})")

# quick test prediction
applicant = pd.DataFrame([{
    "annual_income": 600000, "loan_amount": 200000, "loan_tenure": 36,
    "num_credit_lines": 3, "missed_payments": 1, "employment_years": 4,
    "debt_to_income": 0.30, "credit_score": 720, "age": 29,
    "loan_to_income": 200000 / 600000, "reliability_score": 720 / 2
}])

model = joblib.load("credit_model.pkl")
prob  = model.predict_proba(applicant)[0][1]
label = "CREDITWORTHY ✅" if prob >= 0.5 else "HIGH RISK ❌"

print(f"
applicant: age 29, income ₹6L, loan ₹2L, credit score 720")
print(f"repayment probability: {prob:.2%}")
print(f"decision: {label}")